In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../AirlineScrappedReview_Cleaned_seif_we_ibra#1.csv')

In [ ]:
X = df[['Flying_Date', 'Route', 'Verified', 'Review_title', 'Review_content', 'Traveller_Type', 'Class', 'Start_Location', 'End_Location', 'Layover_Route', 'Start_Longitude', 'Start_Latitude', 'End_Longitude', 'End_Latitude', 'Start_Address', 'End_Address']]

y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Step 1: Feature Engineering for Neural Networks
print("="*80)
print("FEATURE ENGINEERING FOR NEURAL NETWORKS (LIME)")
print("="*80)

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Handle missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Fill numerical missing values with median
for col in numerical_cols:
    median_val = X_train_clean[col].median()
    X_train_clean[col].fillna(median_val, inplace=True)
    X_test_clean[col].fillna(median_val, inplace=True)

# Fill categorical missing values with 'Unknown'
for col in categorical_cols:
    X_train_clean[col].fillna('Unknown', inplace=True)
    X_test_clean[col].fillna('Unknown', inplace=True)

print(f"\n✓ Missing values handled")

# One-Hot Encode categorical features
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_cols, drop_first=True)

# Ensure both sets have the same columns
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns]

print(f"✓ One-hot encoding applied")
print(f"  Features after encoding: {X_train_encoded.shape[1]}")

# IMPORTANT FOR NN: Normalize/Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# Convert back to DataFrames to preserve feature names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns)

print(f"✓ StandardScaler normalization applied")
print(f"  X_train_scaled shape: {X_train_scaled.shape}")
print(f"  X_test_scaled shape: {X_test_scaled.shape}")
print(f"  Feature scaling stats:")
print(f"    - Mean (should be ≈0): {X_train_scaled.mean().mean():.6f}")
print(f"    - Std (should be ≈1): {X_train_scaled.std().mean():.6f}")

feature_names = X_train_scaled.columns.tolist()
class_names = ['Low Rating', 'High Rating']

In [ ]:
# Step 2: Build and Train Neural Network
print("\n" + "="*80)
print("BUILDING AND TRAINING NEURAL NETWORK")
print("="*80)

# Build the neural network
input_dim = X_train_scaled.shape[1]
print(f"\nBuilding Sequential Neural Network...")
print(f"  Input features: {input_dim}")

model = Sequential([
    layers.Dense(128, activation='relu', input_dim=input_dim, name='dense_1'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.Dense(64, activation='relu', name='dense_2'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.Dense(32, activation='relu', name='dense_3'),
    layers.Dense(1, activation='sigmoid', name='output')  # sigmoid for binary classification
])

print("\nModel Architecture:")
model.summary()

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled")

# Train the model with early stopping
print("\nTraining Neural Network...")
print("  (This may take a minute...)")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"✓ Model trained successfully")
print(f"  Total epochs trained: {len(history.history['loss'])}")

# Evaluate model
y_pred_proba = model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
accuracy = accuracy_score(y_test, y_pred)

print(f"\n  Test Accuracy: {accuracy:.4f}")
print(f"  Number of features: {len(feature_names)}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss Over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('lime_nn_training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Training history saved as 'lime_nn_training_history.png'")

In [ ]:
# Step 3: Initialize LIME Explainer for Neural Network
print("\n" + "="*80)
print("LIME EXPLAINER FOR NEURAL NETWORKS")
print("="*80)

print("\nInitializing LIME TabularExplainer (model-agnostic local explanations)...")
print("  • LIME: Local Interpretable Model-agnostic Explanations")
print("  • Approach: Perturbation-based (perturbs inputs locally to understand model)")
print("  • Works with any model (NN, RF, SVM, etc.)")
print("  • Generates local linear approximations around each instance")

# Create prediction function wrapper
def predict_fn(X):
    """Prediction function for LIME - returns probabilities"""
    return model.predict(X, verbose=0)

# Initialize LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,  # Training data for reference
    feature_names=feature_names,
    class_names=class_names,
    mode='regression',  # Use regression for probability output
    verbose=False,
    random_state=42
)

print("\n✓ LIME TabularExplainer created successfully")
print(f"  Explainer type: {type(explainer).__name__}")
print(f"  Number of features: {len(feature_names)}")
print(f"  Training data shape: {X_train_scaled.shape}")
print(f"  Mode: regression (probability output)")

print("\nLIME Configuration:")
print(f"  • Num features: {len(feature_names)}")
print(f"  • Class names: {class_names}")
print(f"  • Perturbation strategy: Random local perturbations")
print(f"  • Model approximation: Local linear model")

In [ ]:
# Step 4: Generate LIME Explanations for Individual Predictions
print("\n" + "="*80)
print("LIME LOCAL EXPLANATIONS - INDIVIDUAL PREDICTIONS")
print("="*80)

# Select samples to explain
sample_indices = [0, 10, 20]
lime_explanations = {}

for idx, sample_idx in enumerate(sample_indices):
    if sample_idx >= len(X_test_scaled):
        print(f"Skipping sample index {sample_idx} (out of range)")
        continue
    
    print(f"\n--- Generating LIME Explanation for Test Sample {sample_idx} ---")
    
    # Get prediction
    instance = X_test_scaled.iloc[sample_idx].values
    nn_pred = model.predict(instance.reshape(1, -1), verbose=0)[0][0]
    actual_rating = y_test.iloc[sample_idx]
    
    print(f"NN Prediction (probability): {nn_pred:.4f}")
    print(f"NN Predicted class: {int(nn_pred > 0.5)} ({class_names[int(nn_pred > 0.5)]})")
    print(f"Actual Rating: {actual_rating}")
    print(f"Correct: {'✓' if (nn_pred > 0.5) == actual_rating else '✗'}")
    
    # Generate LIME explanation
    print(f"\nGenerating local perturbations (1000 samples)...")
    
    exp = explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        num_features=10,  # Show top 10 features
        num_samples=1000   # Generate 1000 perturbations
    )
    
    lime_explanations[sample_idx] = exp
    
    print(f"✓ Explanation generated")
    print(f"\nTop 10 Features Contributing to Prediction:")
    
    # Display feature contributions
    for feat_idx, (feat_name, weight) in enumerate(exp.as_list(), 1):
        direction = '↑ increases' if weight > 0 else '↓ decreases'
        print(f"  {feat_idx:2d}. {feat_name:50s} {direction:15s} ({weight:+.4f})")
    
    # Save visualization
    try:
        fig = exp.as_pyplot_figure()
        plt.tight_layout()
        plt.savefig(f'lime_nn_explanation_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"✓ Explanation visualization saved as 'lime_nn_explanation_sample_{sample_idx}.png'")
    except Exception as e:
        print(f"Note: Could not create figure ({type(e).__name__})")

print("\n" + "="*80)
print("Stored explanations in dict: lime_explanations")
print("Use lime_explanations[sample_idx] to access each explanation")
print("="*80)

In [ ]:
# Step 5: Feature Importance Summary (Aggregate LIME Explanations)
print("\n" + "="*80)
print("AGGREGATE LIME FEATURE IMPORTANCE")
print("="*80)

# Aggregate feature weights across all explanations
feature_weights = {feat: [] for feat in feature_names}

# Sample more predictions for better aggregation
print("\nGenerating LIME explanations for 30 test samples (for aggregation)...")
sample_size = min(30, len(X_test_scaled))

for sample_idx in range(sample_size):
    instance = X_test_scaled.iloc[sample_idx].values
    
    try:
        exp = explainer.explain_instance(
            data_row=instance,
            predict_fn=predict_fn,
            num_features=len(feature_names),  # All features
            num_samples=500  # Fewer samples for speed
        )
        
        # Extract weights
        for feat_name, weight in exp.as_list():
            # Parse feature name to get base feature
            base_feat = feat_name.split(' ')[0]
            if base_feat in feature_weights:
                feature_weights[base_feat].append(abs(weight))
        
        if (sample_idx + 1) % 10 == 0:
            print(f"  Generated {sample_idx + 1}/{sample_size} explanations...")
    except Exception as e:
        print(f"  Warning: Could not explain sample {sample_idx}: {e}")

# Calculate average importance
avg_importance = {}
for feat, weights in feature_weights.items():
    if len(weights) > 0:
        avg_importance[feat] = np.mean(weights)
    else:
        avg_importance[feat] = 0.0

# Sort by importance
sorted_importance = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)

print(f"\n✓ Aggregated LIME importance from {sample_size} explanations")
print(f"\nTop 15 Most Important Features (by average absolute LIME weight):")
print("-" * 70)

for rank, (feat, importance) in enumerate(sorted_importance[:15], 1):
    bar = '█' * int(importance * 50)
    print(f"{rank:2d}. {feat:45s} {importance:7.4f} {bar}")

# Visualize importance
plt.figure(figsize=(12, 8))
top_n = 15
top_features = sorted_importance[:top_n]
feat_names = [f[0] for f in top_features]
feat_values = [f[1] for f in top_features]

plt.barh(range(len(feat_names)), feat_values, color='steelblue')
plt.yticks(range(len(feat_names)), feat_names)
plt.xlabel('Average Absolute LIME Weight')
plt.title('LIME Feature Importance: Top 15 Features (Aggregated from 30 samples)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('lime_nn_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved as 'lime_nn_feature_importance.png'")

In [ ]:
# Step 6: Analyze Feature Impact Patterns
print("\n" + "="*80)
print("LIME FEATURE IMPACT ANALYSIS - POSITIVE VS NEGATIVE WEIGHTS")
print("="*80)

# Track positive and negative impacts
feature_impacts = {feat: {'positive': [], 'negative': []} for feat in feature_names}

print("\nAnalyzing feature impact directions across 20 samples...")
sample_size = min(20, len(X_test_scaled))

for sample_idx in range(sample_size):
    instance = X_test_scaled.iloc[sample_idx].values
    
    try:
        exp = explainer.explain_instance(
            data_row=instance,
            predict_fn=predict_fn,
            num_features=len(feature_names),
            num_samples=500
        )
        
        for feat_name, weight in exp.as_list():
            base_feat = feat_name.split(' ')[0]
            if base_feat in feature_impacts:
                if weight > 0:
                    feature_impacts[base_feat]['positive'].append(weight)
                else:
                    feature_impacts[base_feat]['negative'].append(abs(weight))
    except Exception as e:
        print(f"  Warning at sample {sample_idx}: {e}")

print(f"✓ Analyzed {sample_size} samples")
print(f"\nFeature Impact Directions (Top 10 Features):")
print("-" * 80)

for rank, (feat, importance) in enumerate(sorted_importance[:10], 1):
    pos_weights = feature_impacts[feat]['positive']
    neg_weights = feature_impacts[feat]['negative']
    
    pos_avg = np.mean(pos_weights) if len(pos_weights) > 0 else 0
    neg_avg = np.mean(neg_weights) if len(neg_weights) > 0 else 0
    
    print(f"\n{rank:2d}. {feat}")
    print(f"     Increases prediction (red):   {len(pos_weights):2d} times, avg weight: +{pos_avg:.4f}")
    print(f"     Decreases prediction (blue):  {len(neg_weights):2d} times, avg weight: -{neg_avg:.4f}")

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for ax_idx, (feat, importance) in enumerate(sorted_importance[:6]):
    pos_weights = feature_impacts[feat]['positive']
    neg_weights = feature_impacts[feat]['negative']
    
    ax = axes[ax_idx]
    
    # Create histogram
    if len(pos_weights) > 0:
        ax.hist(pos_weights, bins=10, alpha=0.6, label='Positive (↑)', color='red', edgecolor='black')
    if len(neg_weights) > 0:
        ax.hist(neg_weights, bins=10, alpha=0.6, label='Negative (↓)', color='blue', edgecolor='black')
    
    ax.set_title(f'{feat}\n(Impact Distribution)', fontsize=10, fontweight='bold')
    ax.set_xlabel('LIME Weight')
    ax.set_ylabel('Frequency')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('LIME Feature Impact Distributions (Top 6 Features)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('lime_nn_impact_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Impact distribution plot saved as 'lime_nn_impact_distributions.png'")

In [ ]:
# Step 7: Compare LIME Explanations - Correct vs Incorrect Predictions
print("\n" + "="*80)
print("LIME ANALYSIS: CORRECT VS INCORRECT PREDICTIONS")
print("="*80)

# Identify correct and incorrect predictions
y_pred_all = model.predict(X_test_scaled.values, verbose=0).flatten()
y_pred_class = (y_pred_all > 0.5).astype(int)
is_correct = (y_pred_class == y_test.values)

correct_indices = np.where(is_correct)[0][:10]  # First 10 correct
incorrect_indices = np.where(~is_correct)[0][:10]  # First 10 incorrect

print(f"\nModel Performance:")
print(f"  Correct predictions: {is_correct.sum()} / {len(is_correct)}")
print(f"  Accuracy: {is_correct.sum() / len(is_correct) * 100:.2f}%")

# Analyze correct predictions
correct_weights = {feat: [] for feat in feature_names}
incorrect_weights = {feat: [] for feat in feature_names}

print(f"\nAnalyzing {len(correct_indices)} CORRECT predictions...")
for sample_idx in correct_indices:
    instance = X_test_scaled.iloc[sample_idx].values
    try:
        exp = explainer.explain_instance(instance, predict_fn, num_features=len(feature_names), num_samples=300)
        for feat_name, weight in exp.as_list():
            base_feat = feat_name.split(' ')[0]
            if base_feat in correct_weights:
                correct_weights[base_feat].append(abs(weight))
    except:
        pass

print(f"Analyzing {len(incorrect_indices)} INCORRECT predictions...")
for sample_idx in incorrect_indices:
    instance = X_test_scaled.iloc[sample_idx].values
    try:
        exp = explainer.explain_instance(instance, predict_fn, num_features=len(feature_names), num_samples=300)
        for feat_name, weight in exp.as_list():
            base_feat = feat_name.split(' ')[0]
            if base_feat in incorrect_weights:
                incorrect_weights[base_feat].append(abs(weight))
    except:
        pass

# Compare
print(f"\n✓ Analysis complete")
print(f"\nTop Features Explaining CORRECT vs INCORRECT Predictions:")
print("-" * 80)

correct_importance = {f: np.mean(w) if len(w) > 0 else 0 for f, w in correct_weights.items()}
incorrect_importance = {f: np.mean(w) if len(w) > 0 else 0 for f, w in incorrect_weights.items()}

correct_sorted = sorted(correct_importance.items(), key=lambda x: x[1], reverse=True)[:10]
incorrect_sorted = sorted(incorrect_importance.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 for CORRECT predictions:")
for rank, (feat, imp) in enumerate(correct_sorted, 1):
    print(f"  {rank:2d}. {feat:45s} {imp:.4f}")

print(f"\nTop 10 for INCORRECT predictions:")
for rank, (feat, imp) in enumerate(incorrect_sorted, 1):
    print(f"  {rank:2d}. {feat:45s} {imp:.4f}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Correct predictions
feats_c = [f[0] for f in correct_sorted]
imps_c = [f[1] for f in correct_sorted]
axes[0].barh(range(len(feats_c)), imps_c, color='green', alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(len(feats_c)))
axes[0].set_yticklabels(feats_c, fontsize=9)
axes[0].set_xlabel('Average LIME Weight')
axes[0].set_title(f'Top Features in CORRECT Predictions ({len(correct_indices)} samples)', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

# Incorrect predictions
feats_i = [f[0] for f in incorrect_sorted]
imps_i = [f[1] for f in incorrect_sorted]
axes[1].barh(range(len(feats_i)), imps_i, color='red', alpha=0.7, edgecolor='black')
axes[1].set_yticks(range(len(feats_i)))
axes[1].set_yticklabels(feats_i, fontsize=9)
axes[1].set_xlabel('Average LIME Weight')
axes[1].set_title(f'Top Features in INCORRECT Predictions ({len(incorrect_indices)} samples)', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lime_nn_correct_vs_incorrect.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Comparison plot saved as 'lime_nn_correct_vs_incorrect.png'")

## Summary: LIME Explainability for Neural Networks

### What We've Created (LIME-Specific):

#### **1. LIME Approach (vs SHAP)**
- **Local**: Explains one prediction at a time
- **Interpretable**: Uses simple linear models locally
- **Model-agnostic**: Works with any model (NN, RF, SVM, etc.)
- **Perturbation-based**: Creates synthetic data around the instance

#### **2. Local Explanations (Per-Instance)**
- Generates 1000 local perturbations around each instance
- Fits simple linear model on perturbed data
- Extracts feature weights from local linear model
- Shows which features pushed this specific prediction

#### **3. Aggregate Feature Importance**
- Collects explanations across 30 test samples
- Averages LIME weights across instances
- Provides global importance ranking
- Shows feature consistency across predictions

#### **4. Feature Impact Analysis**
- Separates positive (↑) and negative (↓) impacts
- Shows how often each direction occurs
- Visualizes impact distribution (histogram)
- Reveals feature behavior patterns

#### **5. Prediction Quality Analysis**
- Compares correct vs incorrect predictions
- Shows which features explain successful predictions
- Identifies features in failed predictions
- Helps debug model errors

### SHAP vs LIME for Neural Networks:

| Aspect | SHAP (DeepExplainer) | LIME |
|--------|-------------------|------|
| **Scope** | Global + Local | Local only |
| **Theory** | Shapley values (game theory) | Local linear approximation |
| **Speed** | Gradient-based (fast for NN) | Perturbation-based (slower) |
| **Implementation** | Network-specific | Model-agnostic |
| **Consistency** | Theoretically guaranteed | Not guaranteed |
| **Interpretability** | Can be complex | Simple linear model |
| **Stability** | Depends on model | More stable across instances |
| **Use Case** | Understanding learned patterns | Explaining individual predictions |

### LIME Key Advantages:

✓ **Model-agnostic**: Works with any trained model  
✓ **Locally interpretable**: Simple linear logic per instance  
✓ **Intuitive**: Easy to understand feature contributions  
✓ **No gradient required**: Works with black-box models  
✓ **Trustworthy**: Can verify explanations locally  

### LIME Limitations:

⚠ **Local only**: No global feature importance (need aggregation)  
⚠ **Slow**: Requires 1000 perturbations per instance  
⚠ **Unstable**: Different perturbations may give different results  
⚠ **Kernel width**: Sensitive to hyperparameter choice  
⚠ **High-dimensional**: Less effective with many features  

### How LIME Works (Step-by-Step):

1. **Select instance** to explain (e.g., a test sample)
2. **Create perturbations** by randomly turning features on/off
3. **Get predictions** on perturbed data from the model
4. **Weight perturbations** by distance to original instance
5. **Fit local linear model** on weighted perturbed data
6. **Extract coefficients** as feature importance weights
7. **Visualize** most influential features

### When to Use LIME:

**Best For:**
- Explaining individual predictions to stakeholders
- Debugging specific model errors
- Gaining trust in single decisions
- Working with any black-box model
- Regulatory compliance (simple local explanations)

**Not Ideal For:**
- Understanding global model behavior (use SHAP summary)
- Very large datasets (slow per-instance)
- When speed is critical
- Statistical consistency requirements

### Files Generated:

- `lime_nn_training_history.png` - Training curves
- `lime_nn_explanation_sample_*.png` - Individual explanations
- `lime_nn_feature_importance.png` - Aggregated importance
- `lime_nn_impact_distributions.png` - Impact patterns
- `lime_nn_correct_vs_incorrect.png` - Prediction quality

### Practical Example:

**LIME Explanation Format:**
```
Base prediction: 0.72 (High Rating)
  Route_LHR-JFK:    +0.15  (increases to high rating)
  Review_positive:  +0.12  (pushes toward high rating)
  Price_economy:    -0.08  (reduces rating slightly)
  Class_Business:   +0.18  (major positive factor)
```

This tells you: "This customer got a high rating prediction because it was Business class and mentioned positive review, despite being economy in other bookings."
